# Frente de Redes Neurais – MLP e GRU/LSTM

**Responsável:** Gabriel Modesto · **Plano:** `planos/PLANO_FRENTES_02.11.md` (Seções 3.3 e 4) · **Código comum:** `tt1_comum.py`

| Etapa | O que faz | Saída em `resultados/redes/` |
|---|---|---|
| A | Preparação das bases | – |
| B | Grade candle × janela do MLP na validação (2024): **12 rodadas** | `grade_val.csv`, heatmaps |
| C | Regra de operação na validação | `regras_val.csv` |
| D | Variante: GRU e LSTM (e MLP mais regularizado) na melhor combinação | `variante_val.csv` |
| E | Congelamento e teste (01/2025 → 03/2026), rodado uma vez | `config_final.json`, `teste_final.csv` |
| F | Gráficos finais e resumo | `curva_capital_teste.png`, `previsto_vs_real_teste.png`, `resumo.md` |
| G | Complementar: as 12 combinações do MLP no teste (não muda nenhuma escolha) | `grade_teste_complementar.csv`, `validacao_vs_teste.png` |

Regras seguidas:
- Só o **mini índice (WIN)**. Nenhuma feature do mini dólar entra nos modelos (`COMPLETO` e `LAGS` são só do WIN); o arquivo do WDO é lido apenas porque `tt1_comum.montar_base` o exige.
- Nada de `tt1_comum.py` é alterado. Toda escolha é feita na validação; o teste roda uma vez.
- Critério de escolha: o do **plano comum** (maior esperança por operação com custo de 5 pts; empate → menor Theil U). Como na frente de ML, também reporto o erro-padrão e o **t** de cada esperança e o teste de **Clark-West**, só para leitura: eles não mudam a escolha.

## Etapa A – Preparação

In [ ]:
import os, sys, json, time, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

try:
    import google.colab  # noqa: F401
    PASTA = "/content/data_science_no_mercado_financeiro"
    if not os.path.exists(PASTA):
        subprocess.run(["git", "clone", "https://github.com/anabiaboorges/data_science_no_mercado_financeiro.git", PASTA], check=True)
except ImportError:
    PASTA = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))   # notebook em frentes/redes/

sys.path.append(PASTA)
import tt1_comum as t

FRENTE, MODELO, FEATS = "redes", "mlp", "COMPLETO"
SAIDA = f"{PASTA}/resultados/{FRENTE}"
os.makedirs(f"{SAIDA}/previsoes", exist_ok=True)

win = t.carregar(f"{PASTA}/dados/WINN_M1/WIN$N_M1.csv")
wdo = t.carregar(f"{PASTA}/dados/WDON_M1/WDO$N_M1.csv")   # só para o join do montar_base; não vira feature
BASES = {c: t.montar_base(win, wdo, c) for c in t.CANDLES}
df15, s15 = BASES["15min"]
print(f"Base 15 min: {len(df15):,} linhas, {s15} candles por pregão")   # esperado: ~45 mil e 38
print("Features do MLP:", t.features()[FEATS])
assert not any("wdo" in f for f in t.features()[FEATS])

### Modelos

- **MLP (base, Seção 3.3):** `StandardScaler` + `MLPRegressor((32, 16), alpha=1e-3, early_stopping=True)`, mesmos hiperparâmetros nas 12 combinações.
- **GRU / LSTM (variante):** sequência dos últimos 10 retornos (`LAGS`), uma camada recorrente de 16 unidades, entrada e alvo divididos pelo desvio-padrão do alvo do treino, até 10 épocas com *early stopping* em 10% final do treino. É o esqueleto da Seção 3.3, agora testado.

In [ ]:
import tensorflow as tf
tf.config.experimental.enable_op_determinism()

def mlp(alpha=1e-3):
    return lambda: make_pipeline(StandardScaler(),
                                 MLPRegressor(hidden_layer_sizes=(32, 16), alpha=alpha, early_stopping=True,
                                              max_iter=200, random_state=42))

class RecorrenteT1:
    """GRU/LSTM sobre a sequência dos últimos 10 retornos (ret_lag10 … ret_lag1). Usar com FEATS = "LAGS"."""
    def __init__(self, celula="gru", unidades=16):
        self.celula, self.unidades = celula, unidades
    def _seq(self, X):
        return X[X.columns[::-1]].values[..., None]          # (n, 10, 1), do mais antigo ao mais recente
    def fit(self, X, y):
        Xs = self._seq(X)
        self.esc = y.std()
        tf.keras.utils.set_random_seed(42)
        camada = tf.keras.layers.GRU if self.celula == "gru" else tf.keras.layers.LSTM
        self.m = tf.keras.Sequential([tf.keras.layers.Input((Xs.shape[1], 1)),
                                      camada(self.unidades), tf.keras.layers.Dense(1)])
        self.m.compile(tf.keras.optimizers.Adam(), "mse")
        self.m.fit(Xs / self.esc, y.values / self.esc, epochs=10, batch_size=512, validation_split=0.1,
                   callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=0)
        return self
    def predict(self, X):
        return self.m.predict(self._seq(X) / self.esc, batch_size=4096, verbose=0).ravel() * self.esc

def estat(ops):
    # esperança por operação, erro-padrão e estatística t (só leitura)
    x = ops[ops.lado != 0].pontos
    if len(x) < 30:
        return dict(ep_pts=np.nan, t_stat=np.nan)
    ep = x.std() / np.sqrt(len(x))
    return dict(ep_pts=ep, t_stat=x.mean() / ep)

def clark_west(prev):
    # Clark-West (2007): H0 = o modelo não prevê melhor que o naive (retorno 0). t > 1,645 rejeita a 5%.
    f = prev.y_true ** 2 - (prev.y_true - prev.y_pred) ** 2 + prev.y_pred ** 2
    return f.mean() / (f.std() / np.sqrt(len(f)))

def regras_com_t(prev, custo_pts=5):
    g = t.grade_regras(prev, custo_pts=custo_pts).reset_index(drop=True)
    st = [estat(t.simular(prev, r.limiar, None if pd.isna(r.tp_mult) else r.tp_mult,
                          None if pd.isna(r.sl_mult) else r.sl_mult, custo_pts)) for r in g.itertuples()]
    return pd.concat([g, pd.DataFrame(st)], axis=1)

nn = lambda x: None if pd.isna(x) else float(x)

## Etapa B – Grade candle × janela (validação 2024): as 12 rodadas

MLP com hiperparâmetros fixos em candle 1/15/60 min × janela 1/3/6/12 meses, re-treino trimestral. A esperança aqui é a de operar em todo candle com custo de 5 pts. `dp_prev/dp_real` mostra se a rede prevê magnitude ou fica colada em zero (armadilha da Seção 8).

In [ ]:
FABRICA = mlp()
tab = t.rodar_grade(win, wdo, MODELO, FABRICA, FEATS, SAIDA)

extra = []
for r in tab.itertuples():
    prev = pd.read_csv(f"{SAIDA}/previsoes/WIN_{r.candle}_{r.janela_meses}m_{MODELO}_val.csv", index_col=0, parse_dates=True)
    st = estat(t.simular(prev, custo_pts=5))
    extra.append(dict(dp_prev_sobre_real=prev.y_pred.std() / prev.y_true.std(), cw_t=clark_west(prev),
                      sim_ep_pts=st["ep_pts"], sim_t_stat=st["t_stat"]))
tab = pd.concat([tab, pd.DataFrame(extra)], axis=1)
tab.to_csv(f"{SAIDA}/grade_val.csv", index=False)

t.heatmap(tab, "TheilU", f"{MODELO}: Theil U (validação 2024)", f"{SAIDA}/heatmap_theilU.png", menor_melhor=True)
t.heatmap(tab, "acc_dir", f"{MODELO}: acurácia direcional", f"{SAIDA}/heatmap_acuracia.png", fmt=".3f")
t.heatmap(tab, "sim_esperanca_pts", f"{MODELO}: esperança por operação (pts)", f"{SAIDA}/heatmap_esperanca.png", fmt=".2f")
cols = ["candle", "janela_meses", "tempo_s", "TheilU", "R2_oos", "acc_dir", "dp_prev_sobre_real", "cw_t",
        "sim_n_ops", "sim_esperanca_pts", "sim_ep_pts", "sim_t_stat", "sim_fator_lucro"]
tab[cols].round(4)

In [ ]:
from IPython.display import Image, display
for h in ["heatmap_theilU", "heatmap_acuracia", "heatmap_esperanca"]:
    display(Image(f"{SAIDA}/{h}.png"))

melhor = tab.sort_values(["sim_esperanca_pts", "TheilU"], ascending=[False, True]).iloc[0]
CANDLE, JANELA = melhor.candle, int(melhor.janela_meses)
print(f"Melhor combinação (plano: maior esperança, empate → menor Theil U): candle {CANDLE}, janela {JANELA} m | "
      f"Theil U {melhor.TheilU:.4f} | acurácia {melhor.acc_dir:.1%} | esperança {melhor.sim_esperanca_pts:.2f} pts/op "
      f"(t = {melhor.sim_t_stat:.2f})")
print(f"Combinações com Theil U < 1: {(tab.TheilU < 1).sum()} de {len(tab)}")
print(f"Combinações com esperança > 0: {(tab.sim_esperanca_pts > 0).sum()} de {len(tab)}")

## Etapa C – Regra de operação (validação 2024)

Na melhor combinação: 4 limiares × 3 take profits × 3 stop losses = 36 regras, custo de 5 pontos. Escolhe a de **maior esperança** (critério do plano).

In [ ]:
prev_val = pd.read_csv(f"{SAIDA}/previsoes/WIN_{CANDLE}_{JANELA}m_{MODELO}_val.csv", index_col=0, parse_dates=True)
regras = regras_com_t(prev_val, custo_pts=5)
regras.to_csv(f"{SAIDA}/regras_val.csv", index=False)
regra = regras.iloc[0]
regras.head(10)[["limiar_q", "tp_mult", "sl_mult", "n_ops", "taxa_acerto", "payoff", "esperanca_pts", "ep_pts",
                 "t_stat", "fator_lucro", "resultado_R"]].round(3)

## Etapa D – Variante

Na melhor combinação candle × janela, cada uma com a sua melhor regra (mesmo critério da Etapa C):
1. **GRU** e **LSTM** sobre os últimos 10 retornos (`LAGS`), variante prevista na Seção 3.3.
2. **MLP com `alpha = 0,1`** (regularização 100× maior), sugerido na Seção 8 caso o MLP base fique pior que o naive.
3. **MLP base** para referência.

Se uma variante tiver esperança maior que o MLP base, ela vira o modelo da frente no teste (Seção 4, Etapa D).

In [ ]:
dfb, sb = BASES[CANDLE]
VARIANTES = {"mlp": (mlp(), "COMPLETO"),
             "mlp_alpha0.1": (mlp(0.1), "COMPLETO"),
             "gru": (lambda: RecorrenteT1("gru"), "LAGS"),
             "lstm": (lambda: RecorrenteT1("lstm"), "LAGS")}
PREV_VAR = {}

def avaliar(nome, fabrica, feats_nome):
    t0 = time.time()
    prev = t.walk_forward(dfb, t.features()[feats_nome], fabrica, *t.VALIDACAO, janela_meses=JANELA)
    seg = time.time() - t0
    PREV_VAR[nome] = prev
    prev.to_csv(f"{SAIDA}/previsoes/WIN_{CANDLE}_{JANELA}m_{nome}_val.csv")
    g = regras_com_t(prev, custo_pts=5)
    r = g.iloc[0]
    ops = t.simular(prev, r.limiar, nn(r.tp_mult), nn(r.sl_mult), 5)
    print(f"{nome}: {seg:.0f}s")
    return dict(variante=nome, feats=feats_nome, candle=CANDLE, janela_meses=JANELA, tempo_s=round(seg, 1),
                **t.metricas(prev), dp_prev_sobre_real=prev.y_pred.std() / prev.y_true.std(), cw_t=clark_west(prev),
                limiar_q=r.limiar_q, limiar=r.limiar, tp_mult=nn(r.tp_mult), sl_mult=nn(r.sl_mult),
                **{f"sim_{k}": v for k, v in t.resumo_operacoes(ops).items()},
                **{f"sim_{k}": v for k, v in estat(ops).items()})

var = pd.DataFrame([avaliar(n, f, ft) for n, (f, ft) in VARIANTES.items()]) \
        .sort_values(["sim_esperanca_pts", "TheilU"], ascending=[False, True])
var.to_csv(f"{SAIDA}/variante_val.csv", index=False)
var[["variante", "tempo_s", "TheilU", "R2_oos", "acc_dir", "dp_prev_sobre_real", "cw_t", "limiar_q", "tp_mult",
     "sl_mult", "sim_n_ops", "sim_esperanca_pts", "sim_ep_pts", "sim_t_stat", "sim_fator_lucro"]].round(4)

In [ ]:
final = var.iloc[0]
MODELO_FINAL = final.variante
FABRICA_FINAL, FEATS_FINAL = VARIANTES[MODELO_FINAL]
if MODELO_FINAL != "mlp":
    regra = final                    # a variante vencedora leva a sua própria regra
print(f"Modelo final: {MODELO_FINAL} ({FEATS_FINAL}) | Theil U {final.TheilU:.4f} | "
      f"esperança {final.sim_esperanca_pts:.2f} pts/op (t = {final.sim_t_stat:.2f}) | "
      f"regra: quantil {regra.limiar_q:.0%}, TP {nn(regra.tp_mult)}, SL {nn(regra.sl_mult)}")

## Etapa E – Congelamento e teste (01/2025 → 03/2026)

**Rodado uma única vez.** Depois desta célula, nada volta a ser ajustado.

In [ ]:
tp, sl = nn(regra.tp_mult), nn(regra.sl_mult)
config = dict(frente=FRENTE, modelo=MODELO_FINAL, modelo_base=MODELO, feats=FEATS_FINAL, candle=CANDLE,
              janela_meses=JANELA, limiar_q=float(regra.limiar_q), limiar=float(regra.limiar), tp_mult=tp, sl_mult=sl,
              validacao=list(t.VALIDACAO), teste=list(t.TESTE), custo_escolha_pts=5,
              criterio="plano comum: maior esperança por operação (custo 5 pts); empate → menor Theil U")
json.dump(config, open(f"{SAIDA}/config_final.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps(config, indent=2, ensure_ascii=False))

prev_teste = t.walk_forward(dfb, t.features()[FEATS_FINAL], FABRICA_FINAL, *t.TESTE, janela_meses=JANELA)
prev_teste.to_csv(f"{SAIDA}/previsoes/WIN_{CANDLE}_{JANELA}m_{MODELO_FINAL}_teste.csv")

aleat = t.sinais_aleatorios(prev_teste)
linhas = []
for custo in t.CUSTOS_PTS:
    ops_m = t.simular(prev_teste, regra.limiar, tp, sl, custo)
    ops_a = t.simular(aleat, 0, tp, sl, custo)
    linhas.append(dict(serie="modelo", custo_pts=custo, **t.metricas(prev_teste), cw_t=clark_west(prev_teste),
                       **t.resumo_operacoes(ops_m), **estat(ops_m)))
    linhas.append(dict(serie="aleatorio", custo_pts=custo, **t.resumo_operacoes(ops_a), **estat(ops_a)))
teste = pd.DataFrame(linhas)
teste.to_csv(f"{SAIDA}/teste_final.csv", index=False)
teste[["serie", "custo_pts", "TheilU", "R2_oos", "acc_dir", "cw_t", "n_ops", "taxa_acerto", "payoff",
       "esperanca_pts", "ep_pts", "t_stat", "fator_lucro", "resultado_R"]].round(4)

## Etapa F – Gráficos finais e resumo

In [ ]:
ops_m = t.simular(prev_teste, regra.limiar, tp, sl, 5)
ops_a = t.simular(aleat, 0, tp, sl, 5)
fig, ax = plt.subplots(figsize=(9, 4))
(ops_m.pontos.cumsum() * t.PONTO_WIN).plot(ax=ax, label=f"rede ({MODELO_FINAL})")
(ops_a.pontos.cumsum() * t.PONTO_WIN).plot(ax=ax, label="sinais aleatórios", alpha=0.7)
ax.axhline(0, color="gray", lw=1)
ax.set_ylabel("resultado acumulado (R$, 1 contrato)"); ax.set_xlabel("")
ax.set_title(f"Teste 01/2025–03/2026 · WIN {CANDLE} · janela {JANELA} m · custo 5 pts/op")
ax.legend(); fig.tight_layout(); fig.savefig(f"{SAIDA}/curva_capital_teste.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(prev_teste.y_pred, prev_teste.y_true, s=3, alpha=0.25)
lim = np.quantile(np.abs(prev_teste.y_true), 0.99)
ax.plot([-lim, lim], [-lim, lim], color="red", lw=1, label="previsão perfeita")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("retorno previsto (%)"); ax.set_ylabel("retorno real (%)")
ax.set_title(f"Previsto × real – teste (corr = {prev_teste.y_true.corr(prev_teste.y_pred):.3f})")
ax.legend(); fig.tight_layout(); fig.savefig(f"{SAIDA}/previsto_vs_real_teste.png", dpi=150); plt.show()

In [ ]:
tm = teste[teste.serie == "modelo"].set_index("custo_pts")
ta = teste[teste.serie == "aleatorio"].set_index("custo_pts")
fmt = lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
fmtR = lambda x: f"{x:,.0f}".replace(",", ".")
cw = tm.loc[5, "cw_t"]
bate = "bate" if tm.loc[5, "TheilU"] < 1 else "não bate"
cw_txt = (f"Clark-West t = {cw:.2f}: a previsão é estatisticamente melhor que o naive" if cw > 1.645
          else f"Clark-West t = {cw:.2f}: sem evidência de previsão melhor que o naive")
lucra = tm.loc[5, "esperanca_pts"] > 0 and tm.loc[5, "fator_lucro"] > 1
corr_acc_esp = tab[["acc_dir", "sim_esperanca_pts"]].corr(method="spearman").iloc[0, 1]
desc = lambda x: "sem" if x is None else f"{x:g}× a volatilidade"
var_txt = "; ".join(f"{r.variante} {r.sim_esperanca_pts:.2f}" for r in var.itertuples())
dp = prev_teste.y_pred.std() / prev_teste.y_true.std()
esp = " / ".join(fmt(tm.loc[c, "esperanca_pts"]) for c in t.CUSTOS_PTS)

resumo = f'''# Resumo – Frente de redes neurais (MLP / GRU / LSTM)

- **Q1 – Candle e janela:** nas 12 rodadas do MLP na validação 2024, a melhor combinação foi candle de **{CANDLE}** com janela de **{JANELA} {"mês" if JANELA == 1 else "meses"}** (esperança {fmt(melhor.sim_esperanca_pts)} pts/op operando sempre, t = {melhor.sim_t_stat:.2f}; Theil U {melhor.TheilU:.4f}). {(tab.TheilU < 1).sum()} de 12 combinações tiveram Theil U < 1 e {(tab.sim_esperanca_pts > 0).sum()} de 12 tiveram esperança positiva. Na variante (esperança com a melhor regra, pts/op: {var_txt}) o modelo final ficou **{MODELO_FINAL}**.
- **Q2 – Bate o naive?** No teste, Theil U = {tm.loc[5, "TheilU"]:.4f} e R²_oos = {tm.loc[5, "R2_oos"]:.5f}: {bate} o naive pelo erro ({cw_txt}). O desvio da previsão é {dp:.0%} do desvio do retorno real.
- **Q3 – Acertar mais é ganhar mais?** Acurácia direcional no teste de {tm.loc[5, "acc_dir"]:.1%} e taxa de acerto das operações de {tm.loc[5, "taxa_acerto"]:.1%} (payoff {tm.loc[5, "payoff"]:.2f}); na grade de validação, a correlação de Spearman entre acurácia e esperança foi {corr_acc_esp:.2f}.
- **Q4 – Regra e teste:** regra escolhida na validação: limiar no quantil {regra.limiar_q:.0%} de |previsão|, take profit {desc(tp)}, stop loss {desc(sl)}. No teste, esperança de {esp} pts/op com custo 0 / 5 / 10 pts (t = {tm.loc[5, "t_stat"]:.2f} com 5 pts), fator de lucro {tm.loc[5, "fator_lucro"]:.3f} e resultado de R$ {fmtR(tm.loc[5, "resultado_R"])} em {int(tm.loc[5, "n_ops"]):,} operações (aleatório: {fmt(ta.loc[5, "esperanca_pts"])} pts/op). {"É lucrativa" if lucra else "Não é lucrativa"} com custo de 5 pts.
- **Limitação/surpresa:** ver a seção de análise abaixo e `relatorio.md`.
'''
print(resumo)

## Etapa G – Complementar: as 12 combinações do MLP no teste

**Rodada depois do teste oficial, não muda nenhuma escolha.** Serve só para ver se o ranking da validação se repete fora da amostra (mesma análise que a frente de ML fez em `grade_teste_complementar.csv`).

In [ ]:
tab_teste = t.rodar_grade(win, wdo, "mlp_complementar", FABRICA, FEATS, SAIDA, periodo=t.TESTE)
tab_teste.to_csv(f"{SAIDA}/grade_teste_complementar.csv", index=False)
os.remove(f"{SAIDA}/grade_teste.csv")          # o rodar_grade grava com nome genérico; fica só o complementar
comp = tab[["candle", "janela_meses", "TheilU", "sim_esperanca_pts"]].merge(
    tab_teste[["candle", "janela_meses", "TheilU", "sim_esperanca_pts"]], on=["candle", "janela_meses"], suffixes=("_val", "_teste"))
rho_u = comp[["TheilU_val", "TheilU_teste"]].corr(method="spearman").iloc[0, 1]
rho_e = comp[["sim_esperanca_pts_val", "sim_esperanca_pts_teste"]].corr(method="spearman").iloc[0, 1]

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, nome, rho in [(axs[0], "TheilU", "Theil U", rho_u), (axs[1], "sim_esperanca_pts", "Esperança (pts/op, opera sempre)", rho_e)]:
    for r in comp.itertuples():
        x, y = getattr(r, f"{col}_val"), getattr(r, f"{col}_teste")
        ax.scatter(x, y, color={"1min": "tab:blue", "15min": "tab:orange", "60min": "tab:green"}[r.candle])
        ax.annotate(f"{r.candle} {r.janela_meses}m", (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("validação 2024"); ax.set_ylabel("teste 2025–03/2026"); ax.set_title(f"{nome} · Spearman {rho:.2f}")
fig.suptitle("MLP: o ranking da validação se repete no teste?"); fig.tight_layout()
fig.savefig(f"{SAIDA}/validacao_vs_teste.png", dpi=150); plt.show()
comp.round(4)

## Análise

In [ ]:
# Números usados no texto da análise e no relatorio.md
gs = tab.groupby("candle")[["TheilU", "acc_dir", "sim_esperanca_pts"]].mean().reindex(t.CANDLES)
print("Médias por candle (validação):"); print(gs.round(4))
print(f"\nSpearman validação × teste: Theil U {rho_u:.2f} | esperança {rho_e:.2f}")
print(f"Ganho bruto por operação no teste (custo 0): {tm.loc[0, 'esperanca_pts']:.2f} pts")